# Ноутбук 02. BERT-векторизація шаблонів через токен [CLS]

Цей ноутбук перетворює Drain-шаблони, отримані в ноутбуці 01, у 768-вимірні вектори за допомогою попередньо натренованої моделі `bert-base-uncased`. Для кожного шаблону беремо ембединг спеціального токена `[CLS]` з останнього прихованого стану — це стандартне зведене подання послідовності в BERT (див. розділ 1.7 тези, формула `eq:cls_template`). Перед векторизацією ми розширюємо словник токенізатора плейсхолдерами регекс-нормалізації (`<IP>`, `<NUM>`, `<PATH>`, `<HEX>`, `<EXC>`, `<SESSION>`, `<UUID>`), щоб зберегти семантичну атомарність цих сутностей: без цього розширення BERT розіб'є, наприклад, `<NUM>` на чотири фрагменти, втрачаючи саму ідею нормалізованого плейсхолдера. Векторизуються лише унікальні шаблони (а не сирі записи), що скорочує обчислення з десятків тисяч рядків до десятків шаблонів — це і є ключове оптимізаційне рішення гібридного методу.

In [1]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import numpy as np
import torch
from transformers import BertTokenizer

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PROCESSED = ROOT / 'data' / 'processed'
TEMPLATES_DIR = DATA_PROCESSED / 'templates'
EMBEDDINGS_DIR = DATA_PROCESSED / 'embeddings'
TOKENIZER_DIR = DATA_PROCESSED / 'tokenizer'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s — %(message)s',
    force=True,
)
logging.getLogger('transformers').setLevel(logging.WARNING)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('huggingface_hub').setLevel(logging.WARNING)

torch.manual_seed(42)
np.random.seed(42)

from src.bert.vectorizer import TemplateVectorizer
from src.io.persistence import load_json, save_json, save_numpy

DEVICE = 'cpu'
print(f'torch.cuda.is_available() = {torch.cuda.is_available()}')
print(f'device = {DEVICE}')
print(f'torch = {torch.__version__}')
print(f'numpy = {np.__version__}')


/Users/roman/Personal/dyploma/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.cuda.is_available() = False
device = cpu
torch = 2.11.0
numpy = 2.4.4


## 1. Завантаження шаблонів з ноутбука 01

Спочатку перевіряємо, що JSON-файли з шаблонами існують та містять очікувану кількість записів.

In [2]:
templates_2k_path = TEMPLATES_DIR / 'zookeeper_2k_templates.json'
templates_full_path = TEMPLATES_DIR / 'zookeeper_full_templates.json'

for path in (templates_2k_path, templates_full_path):
    assert path.exists(), f'Missing prerequisite file: {path}. Run notebook 01 first.'

templates_2k = load_json(templates_2k_path)
templates_full = load_json(templates_full_path)

assert len(templates_2k) >= 30, f'2k file should have >=30 templates, got {len(templates_2k)}'
assert len(templates_full) >= 50, f'full file should have >=50 templates, got {len(templates_full)}'

print(f'2k templates  : {len(templates_2k)}')
print(f'full templates: {len(templates_full)}')
print()
print('Top-5 templates from the full dataset (by support):')
for t in templates_full[:5]:
    print(f"  id={t['id']:>3}  support={t['support']:>6}  {t['template']}")


2k templates  : 46
full templates: 77

Top-5 templates from the full dataset (by support):
  id= 20  support= 10429  Received connection request /<IP>
  id= 31  support= 10390  Interrupting SendWorker
  id= 33  support= 10390  Send worker leaving thread
  id= 30  support= 10388  Connection broken for id <NUM>, my id = <NUM>, error =
  id= 32  support= 10386  Interrupted while waiting for message on queue


## 2. Проблема стандартного токенізатора

За замовчуванням токенізатор `bert-base-uncased` нічого не знає про наші плейсхолдери. Він використовує алгоритм WordPiece, який розбиває невідомі рядки на фрагменти субсловника. Через це `<NUM>`, `<IP>` та інші маркери перетворюються на послідовність із 3–5 токенів з префіксами `##`, повністю втрачаючи свою роль атомарної сутності. Це і є той ефект, який ми зараз продемонструємо, перш ніж усувати його.

In [3]:
plain_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Pick three representative templates that contain different placeholders.
demo_templates = [
    next(t['template'] for t in templates_full if '<IP>' in t['template']),
    next(t['template'] for t in templates_full if '<NUM>' in t['template']),
    next(t['template'] for t in templates_full if '<EXC>' in t['template']),
]

print('WordPiece breakdown WITHOUT tokenizer extension:')
print('-' * 70)
for template in demo_templates:
    tokens = plain_tokenizer.tokenize(template)
    print(f'  TEMPLATE: {template}')
    print(f'  TOKENS  : {tokens}')
    print()


WordPiece breakdown WITHOUT tokenizer extension:
----------------------------------------------------------------------
  TEMPLATE: Received connection request /<IP>
  TOKENS  : ['received', 'connection', 'request', '/', '<', 'ip', '>']

  TEMPLATE: Connection broken for id <NUM>, my id = <NUM>, error =
  TOKENS  : ['connection', 'broken', 'for', 'id', '<', 'nu', '##m', '>', ',', 'my', 'id', '=', '<', 'nu', '##m', '>', ',', 'error', '=']

  TEMPLATE: Exception causing close of session <SESSION> due to <EXC>: ZooKeeperServer not running
  TOKENS  : ['exception', 'causing', 'close', 'of', 'session', '<', 'session', '>', 'due', 'to', '<', 'ex', '##c', '>', ':', 'zoo', '##keepers', '##er', '##ver', 'not', 'running']



## 3. Розширення словника токенізатора плейсхолдерами

Виправлення складається з двох частин. По-перше, ми додаємо плейсхолдери як *special tokens* у словник токенізатора (`add_special_tokens`) і збільшуємо матрицю ембедингів моделі (`resize_token_embeddings`). По-друге — і це принципово важливо — нові рядки матриці ембедингів ініціалізуються середнім значенням усіх існуючих ембедингів, а не випадково. Випадкова ініціалізація залишила б ці токени без будь-якого семантичного якоря, через що вектори шаблонів з плейсхолдерами були б хаотичними. Середній ембединг — дешевий, але набагато якісніший приор.

In [4]:
vectorizer = TemplateVectorizer(device=DEVICE)

print(f'vocab size BEFORE extension: {vectorizer.vocab_size_before}')
print(f'vocab size AFTER  extension: {vectorizer.vocab_size_after}')
print(f'newly added token count    : {vectorizer.num_added_tokens}')
print(f'new token ids              : {vectorizer.new_token_ids}')
print()
print('Side-by-side tokenization (before → after extension):')
print('-' * 70)
for template in demo_templates:
    before = plain_tokenizer.tokenize(template)
    after = vectorizer.get_tokenization(template)
    print(f'  TEMPLATE: {template}')
    print(f'  BEFORE  : {before}')
    print(f'  AFTER   : {after}')
    print()


2026-05-12 19:44:46,084 WARNING huggingface_hub.utils._http — Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14544.27it/s]


[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


vocab size BEFORE extension: 30522
vocab size AFTER  extension: 30529
newly added token count    : 7
new token ids              : [30522, 30523, 30524, 30525, 30526, 30527, 30528]

Side-by-side tokenization (before → after extension):
----------------------------------------------------------------------
  TEMPLATE: Received connection request /<IP>
  BEFORE  : ['received', 'connection', 'request', '/', '<', 'ip', '>']
  AFTER   : ['received', 'connection', 'request', '/', '<IP>']

  TEMPLATE: Connection broken for id <NUM>, my id = <NUM>, error =
  BEFORE  : ['connection', 'broken', 'for', 'id', '<', 'nu', '##m', '>', ',', 'my', 'id', '=', '<', 'nu', '##m', '>', ',', 'error', '=']
  AFTER   : ['connection', 'broken', 'for', 'id', '<NUM>', ',', 'my', 'id', '=', '<NUM>', ',', 'error', '=']

  TEMPLATE: Exception causing close of session <SESSION> due to <EXC>: ZooKeeperServer not running
  BEFORE  : ['exception', 'causing', 'close', 'of', 'session', '<', 'session', '>', 'due', 'to', '<'

## 4. Перевірка коректності токенізації

Перевіряємо інваріант: рядок `"X <PLACEHOLDER> Y"` повинен токенізуватися рівно в три токени — `"x"`, плейсхолдер як одне ціле, та `"y"`. Якщо це виконується для всіх семи плейсхолдерів, розширення працює правильно.

In [5]:
for placeholder in TemplateVectorizer.PLACEHOLDER_TOKENS:
    tokens = vectorizer.get_tokenization(f'X {placeholder} Y')
    expected = ['x', placeholder, 'y']
    assert tokens == expected, f'Tokenization failure for {placeholder}: got {tokens}'

print('OK — всі плейсхолдери токенізуються атомарно')


OK — всі плейсхолдери токенізуються атомарно


## 5. Векторизація шаблонів 2k-вибірки

Беремо рядки шаблонів з 2k-файлу і прогоняємо їх через `vectorize_batch`. Очікувана форма вихідного масиву: `(46, 768)`, dtype `float32`. Статистика по матриці має бути в межах, типових для BERT [CLS] (середнє близько 0, стандартне відхилення приблизно 0.3–0.5).

In [6]:
template_strings_2k = [t['template'] for t in templates_2k]
vectors_2k = vectorizer.vectorize_batch(template_strings_2k, batch_size=32)

assert vectors_2k.shape == (len(templates_2k), 768), (
    f'Expected ({len(templates_2k)}, 768), got {vectors_2k.shape}'
)
assert vectors_2k.dtype == np.float32, f'Expected float32, got {vectors_2k.dtype}'

print(f'2k vectors shape: {vectors_2k.shape}')
print(f'  mean : {vectors_2k.mean():+.4f}')
print(f'  std  : {vectors_2k.std():+.4f}')
print(f'  min  : {vectors_2k.min():+.4f}')
print(f'  max  : {vectors_2k.max():+.4f}')


vectorize_batch:   0%|          | 0/2 [00:00<?, ?it/s]

vectorize_batch:  50%|█████     | 1/2 [00:01<00:01,  1.38s/it]

vectorize_batch: 100%|██████████| 2/2 [00:01<00:00,  1.46it/s]

vectorize_batch: 100%|██████████| 2/2 [00:01<00:00,  1.27it/s]

2k vectors shape: (46, 768)
  mean : -0.0088
  std  : +0.5074
  min  : -9.6372
  max  : +4.1724


In [7]:
vectors_2k_path = EMBEDDINGS_DIR / 'zookeeper_2k_embeddings.npy'
mapping_2k_path = EMBEDDINGS_DIR / 'zookeeper_2k_id_mapping.json'

save_numpy(vectors_2k, vectors_2k_path)

mapping_2k = [
    {
        'row_index': i,
        'template_id': t['id'],
        'template': t['template'],
        'support': t['support'],
    }
    for i, t in enumerate(templates_2k)
]
save_json(mapping_2k, mapping_2k_path)

print(f'Saved: {vectors_2k_path}  ({vectors_2k_path.stat().st_size:,} bytes)')
print(f'Saved: {mapping_2k_path}  ({mapping_2k_path.stat().st_size:,} bytes)')


Saved: /Users/roman/Personal/dyploma/data/processed/embeddings/zookeeper_2k_embeddings.npy  (141,440 bytes)
Saved: /Users/roman/Personal/dyploma/data/processed/embeddings/zookeeper_2k_id_mapping.json  (6,342 bytes)


## 6. Sanity-check: семантична близькість шаблонів

Якість векторизації грубо перевіряється косинусною подібністю на ручно підібраних парах: дві пари семантично близьких шаблонів мають давати вищі значення, ніж пара семантично далеких. Це не математичний доказ якості, а швидка sanity-перевірка, яка ловить грубі помилки (наприклад, якщо модель чи токенізатор завантажилися неправильно).

In [8]:
def find_template(substring: str, templates: list[dict]) -> str:
    """Return the first template whose string contains `substring`."""
    for t in templates:
        if substring in t['template']:
            return t['template']
    raise ValueError(f'No template contains substring: {substring!r}')

# Pair A — both about quorum/worker thread lifecycle.
pair_a = (
    find_template('Send worker leaving thread', templates_2k),
    find_template('Interrupted while waiting for message', templates_2k),
)
# Pair B — both about incoming connection events.
pair_b = (
    find_template('Received connection request', templates_2k),
    find_template('Accepted socket connection', templates_2k),
)
# Pair C — connection event vs snapshot read (semantically distant).
pair_c = (
    find_template('Received connection request', templates_2k),
    find_template('Reading snapshot', templates_2k),
)

pairs = [('A (worker lifecycle)', pair_a), ('B (connections)', pair_b), ('C (connection vs snapshot)', pair_c)]
scores: dict[str, float] = {}
for label, (a, b) in pairs:
    score = vectorizer.cosine_similarity(a, b)
    scores[label] = score
    print(f'Pair {label}')
    print(f'  A: {a}')
    print(f'  B: {b}')
    print(f'  cosine similarity: {score:+.4f}')
    print()

score_a = scores['A (worker lifecycle)']
score_b = scores['B (connections)']
score_c = scores['C (connection vs snapshot)']
if score_a < score_c or score_b < score_c:
    print(
        'WARNING: expected pairs A and B (semantically close) to score higher '
        f'than pair C. Got A={score_a:.3f}, B={score_b:.3f}, C={score_c:.3f}. '
        'Inspect manually — model is NOT being changed automatically.'
    )
else:
    print('Sanity-check passed: close pairs score higher than the distant pair.')


Pair A (worker lifecycle)
  A: Send worker leaving thread
  B: Interrupted while waiting for message on queue
  cosine similarity: +0.8494

Pair B (connections)
  A: Received connection request /<IP>
  B: Accepted socket connection from /<IP>
  cosine similarity: +0.9588



Pair C (connection vs snapshot)
  A: Received connection request /<IP>
  B: Reading snapshot <PATH>
  cosine similarity: +0.9109



## 7. Векторизація шаблонів повної версії

Та сама процедура для 77 шаблонів повного датасету. Очікувана форма: `(77, 768)`.

In [9]:
template_strings_full = [t['template'] for t in templates_full]
vectors_full = vectorizer.vectorize_batch(template_strings_full, batch_size=32)

assert vectors_full.shape == (len(templates_full), 768), (
    f'Expected ({len(templates_full)}, 768), got {vectors_full.shape}'
)
assert vectors_full.dtype == np.float32

print(f'full vectors shape: {vectors_full.shape}')
print(f'  mean : {vectors_full.mean():+.4f}')
print(f'  std  : {vectors_full.std():+.4f}')
print(f'  min  : {vectors_full.min():+.4f}')
print(f'  max  : {vectors_full.max():+.4f}')

vectors_full_path = EMBEDDINGS_DIR / 'zookeeper_full_embeddings.npy'
mapping_full_path = EMBEDDINGS_DIR / 'zookeeper_full_id_mapping.json'

save_numpy(vectors_full, vectors_full_path)
mapping_full = [
    {
        'row_index': i,
        'template_id': t['id'],
        'template': t['template'],
        'support': t['support'],
    }
    for i, t in enumerate(templates_full)
]
save_json(mapping_full, mapping_full_path)

print(f'Saved: {vectors_full_path}  ({vectors_full_path.stat().st_size:,} bytes)')
print(f'Saved: {mapping_full_path}  ({mapping_full_path.stat().st_size:,} bytes)')


vectorize_batch:   0%|          | 0/3 [00:00<?, ?it/s]

vectorize_batch:  33%|███▎      | 1/3 [00:00<00:00,  2.49it/s]

vectorize_batch:  67%|██████▋   | 2/3 [00:00<00:00,  2.54it/s]

vectorize_batch: 100%|██████████| 3/3 [00:00<00:00,  3.41it/s]

full vectors shape: (77, 768)
  mean : -0.0089
  std  : +0.5091
  min  : -9.6372
  max  : +4.1724
Saved: /Users/roman/Personal/dyploma/data/processed/embeddings/zookeeper_full_embeddings.npy  (236,672 bytes)
Saved: /Users/roman/Personal/dyploma/data/processed/embeddings/zookeeper_full_id_mapping.json  (10,574 bytes)


## 8. Експорт стану токенізатора

Зберігаємо розширений токенізатор у `data/processed/tokenizer/`, щоб ноутбук 05 (візуалізація attention) міг відтворити рівно ту саму токенізацію без повторної ініціалізації.

In [10]:
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
vectorizer.tokenizer.save_pretrained(str(TOKENIZER_DIR))

saved_files = sorted(TOKENIZER_DIR.iterdir())
total_size = sum(f.stat().st_size for f in saved_files if f.is_file())
print(f'Saved tokenizer to: {TOKENIZER_DIR}')
print(f'Files ({len(saved_files)}):')
for f in saved_files:
    if f.is_file():
        print(f'  {f.name:<30}  {f.stat().st_size:>10,} bytes')
print(f'Total size: {total_size:,} bytes')


Saved tokenizer to: /Users/roman/Personal/dyploma/data/processed/tokenizer
Files (2):
  tokenizer.json                     712,921 bytes
  tokenizer_config.json                  479 bytes
Total size: 713,400 bytes


## Висновки

Розширили словник токенізатора `bert-base-uncased` сімома плейсхолдерами регекс-нормалізації та ініціалізували їхні ембединги середнім значенням існуючої матриці. Векторизували всі 46 шаблонів 2k-вибірки та всі 77 шаблонів повного датасету в 768-вимірні вектори [CLS]; статистика матриць (середнє, std) лежить у типових для BERT межах. Sanity-check на трьох ручно підібраних парах підтвердив, що шаблони з однієї семантичної області (життєвий цикл воркерів, події підключення) ближчі один до одного, ніж до подій іншого типу (читання снапшота). Усі артефакти збережено у `data/processed/embeddings/` та `data/processed/tokenizer/` і готові до споживання ноутбуками 03 (KMeans-кластеризація) і 04 (класифікація).

In [11]:
print('=' * 70)
print('DELIVERABLE SUMMARY')
print('=' * 70)
print()
print('Files created or modified:')
for p in [
    ROOT / 'src' / 'io' / 'persistence.py',
    ROOT / 'src' / 'bert' / 'vectorizer.py',
    ROOT / 'notebooks' / '02_bert_vectorization.ipynb',
    vectors_2k_path,
    mapping_2k_path,
    vectors_full_path,
    mapping_full_path,
    TOKENIZER_DIR,
]:
    print(f'  {p}')
print()
print(f'Tokenizer vocab size: {vectorizer.vocab_size_before} → {vectorizer.vocab_size_after} (+{vectorizer.num_added_tokens})')
print(f'Templates vectorized (2k)  : {vectors_2k.shape[0]}')
print(f'Templates vectorized (full): {vectors_full.shape[0]}')
print()
print('Cosine similarity sanity-check:')
for label, score in scores.items():
    print(f'  Pair {label}: {score:+.4f}')
print()
print('Suggested commit:')
print('  git add src/bert/vectorizer.py src/io/persistence.py \\')
print('          notebooks/02_bert_vectorization.ipynb \\')
print('          data/processed/embeddings data/processed/tokenizer')
print('  git commit -m "feat(notebook-02): bert vectorization of drain templates"')


DELIVERABLE SUMMARY

Files created or modified:
  /Users/roman/Personal/dyploma/src/io/persistence.py
  /Users/roman/Personal/dyploma/src/bert/vectorizer.py
  /Users/roman/Personal/dyploma/notebooks/02_bert_vectorization.ipynb
  /Users/roman/Personal/dyploma/data/processed/embeddings/zookeeper_2k_embeddings.npy
  /Users/roman/Personal/dyploma/data/processed/embeddings/zookeeper_2k_id_mapping.json
  /Users/roman/Personal/dyploma/data/processed/embeddings/zookeeper_full_embeddings.npy
  /Users/roman/Personal/dyploma/data/processed/embeddings/zookeeper_full_id_mapping.json
  /Users/roman/Personal/dyploma/data/processed/tokenizer

Tokenizer vocab size: 30522 → 30529 (+7)
Templates vectorized (2k)  : 46
Templates vectorized (full): 77

Cosine similarity sanity-check:
  Pair A (worker lifecycle): +0.8494
  Pair B (connections): +0.9588
  Pair C (connection vs snapshot): +0.9109

Suggested commit:
  git add src/bert/vectorizer.py src/io/persistence.py \
          notebooks/02_bert_vectorizati